# json module
- **JSON** (JavaScript Object Notation) is a text format for storing and sending data. It's everywhere: config files, web APIs, saving results/metrics, datasets.
- The `json` module converts Python objects ⇄ JSON text.

### Python ⇄ JSON types
| Python | JSON |
|---|---|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string (always **double quotes** `"..."`) |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

⚠️ JSON keys are always **strings**: `{1: "a"}` becomes `{"1": "a"}`. A `tuple` comes back as a `list`.

### The 4 main functions ⭐
| Function | Direction | Works with |
|---|---|---|
| `json.dumps(obj)` | Python → JSON | **string** (the `s` = string) |
| `json.loads(text)` | JSON → Python | **string** |
| `json.dump(obj, f)` | Python → JSON | **file** |
| `json.load(f)` | JSON → Python | **file** |

In [ ]:
import json

### 1. `dumps` — Python → JSON string
Useful options:
- `indent=4` → pretty, readable output
- `sort_keys=True` → keys in alphabetical order
- `ensure_ascii=False` → keep non-English characters (e.g. Arabic) instead of `م...`

In [ ]:
data = {"name": "Mahmoud", "age": 25, "skills": ["python", "ml"], "active": True, "score": None}

text = json.dumps(data)
print(text, type(text))

print(json.dumps(data, indent=4, sort_keys=True))
print(json.dumps({"city": "القاهرة"}))
print(json.dumps({"city": "القاهرة"}, ensure_ascii=False))

### 2. `loads` — JSON string → Python

In [ ]:
text = '{"model": "resnet", "layers": 50, "pretrained": true, "dropout": null}'
obj = json.loads(text)
print(obj, type(obj))
print(obj["layers"] + 1)
print(json.loads(json.dumps({1: "a", "t": (1, 2)})))   # int key -> "1", tuple -> list

### 3. `dump` / `load` — working with files ⭐
Most common usage: save & load config files, results, metrics.

In [ ]:
config = {"lr": 0.001, "epochs": 10, "batch_size": 32, "layers": [64, 32]}

with open("config.json", "w") as f:
    json.dump(config, f, indent=4)

with open("config.json") as f:
    loaded = json.load(f)

print(loaded)
print(loaded == config)

With `pathlib` it's a one-liner:

In [ ]:
from pathlib import Path

p = Path("config.json")
p.write_text(json.dumps(config, indent=4))
print(json.loads(p.read_text()))
p.unlink()   # cleanup

### 4. Common errors
| Error | Why | Fix |
|---|---|---|
| `json.JSONDecodeError` | text is not valid JSON (single quotes, trailing comma, `True` instead of `true`) | fix the text / `try-except` |
| `TypeError: ... is not JSON serializable` | object type JSON doesn't know (`set`, `datetime`, custom class) | convert first, or use `default=str` |

In [ ]:
try:
    json.loads("{'name': 'bad'}")     # single quotes are NOT valid JSON
except json.JSONDecodeError as e:
    print("invalid JSON:", e)

In [ ]:
from datetime import datetime

info = {"tags": {"a", "b"}, "created": datetime(2026, 11, 15)}
try:
    json.dumps(info)
except TypeError as e:
    print("error:", e)

print(json.dumps({"tags": sorted(info["tags"]), "created": info["created"]}, default=str))

## 🏋️ Exercises
- Try each exercise yourself in the empty cell first, then open **💡 Show solution** to compare.
- Run the **setup** cell first.
- If something breaks, just re-run the setup cell to reset everything.

In [ ]:
# SETUP: creates a practice folder (re-run anytime to reset)
import json, shutil
from pathlib import Path

base = Path("ex_json")
shutil.rmtree(base, ignore_errors=True)
base.mkdir()

(base / "config.json").write_text(json.dumps(
    {"model": "cnn", "lr": 0.01, "epochs": 5, "optimizer": {"name": "adam", "beta": 0.9}}, indent=4))

(base / "users.json").write_text(json.dumps([
    {"name": "Ali", "age": 22, "city": "Cairo"},
    {"name": "Sara", "age": 31, "city": "Alex"},
    {"name": "Omar", "age": 27, "city": "Cairo"},
    {"name": "Mona", "age": 19, "city": "Giza"},
], indent=4))
print("setup done:", sorted(p.name for p in base.iterdir()))

### Exercise 1
**Round trip** — convert `student = {"name": "Ali", "grades": [90, 85], "passed": True}` to a pretty JSON string (indent 2) and print it.
Then convert it back to Python and check it equals the original.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
student = {"name": "Ali", "grades": [90, 85], "passed": True}
text = json.dumps(student, indent=2)
print(text)
back = json.loads(text)
print(back == student)
```
</details>

### Exercise 2
**Update a config file** — load `ex_json/config.json`, change `epochs` to `20` and the optimizer name to `"sgd"`, then save it back (pretty).
Load it again to confirm the change.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
path = base / "config.json"
with open(path) as f:
    config = json.load(f)

config["epochs"] = 20
config["optimizer"]["name"] = "sgd"

with open(path, "w") as f:
    json.dump(config, f, indent=4)

with open(path) as f:
    print(json.load(f))
```
</details>

### Exercise 3
**Query a JSON list** — load `ex_json/users.json` and print:
1. the names of users older than 25
2. the average age
3. how many users live in Cairo

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "users.json") as f:
    users = json.load(f)

print([u["name"] for u in users if u["age"] > 25])
print(sum(u["age"] for u in users) / len(users))
print(sum(1 for u in users if u["city"] == "Cairo"))
```
</details>

### Exercise 4
**Safe loading** — write a function `safe_loads(text)` that returns the parsed object, or `None` if the text isn't valid JSON.
Test it with `'{"a": 1}'`, `"{'a': 1}"` and `'[1, 2,]'`.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
def safe_loads(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print(safe_loads('{"a": 1}'))
print(safe_loads("{'a': 1}"))
print(safe_loads('[1, 2,]'))
```
</details>

### Exercise 5
**Save training history** — you trained a model for 4 epochs:
```python
history = [
    {"epoch": 1, "loss": 0.9, "val_acc": 0.61},
    {"epoch": 2, "loss": 0.6, "val_acc": 0.72},
    {"epoch": 3, "loss": 0.4, "val_acc": 0.78},
    {"epoch": 4, "loss": 0.3, "val_acc": 0.75},
]
```
Save it to `ex_json/history.json`, load it back, and print the epoch with the best `val_acc`.
Hint: `max(..., key=...)`

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
history = [
    {"epoch": 1, "loss": 0.9, "val_acc": 0.61},
    {"epoch": 2, "loss": 0.6, "val_acc": 0.72},
    {"epoch": 3, "loss": 0.4, "val_acc": 0.78},
    {"epoch": 4, "loss": 0.3, "val_acc": 0.75},
]
with open(base / "history.json", "w") as f:
    json.dump(history, f, indent=4)

with open(base / "history.json") as f:
    loaded = json.load(f)

best = max(loaded, key=lambda h: h["val_acc"])
print(best["epoch"], best["val_acc"])
```
</details>

### Exercise 6
**Not serializable** — make this work with `json.dumps` (without removing any data):
```python
from datetime import date
record = {"id": 7, "tags": {"ml", "python"}, "day": date(2026, 11, 15)}
```

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
from datetime import date
record = {"id": 7, "tags": {"ml", "python"}, "day": date(2026, 11, 15)}

# option 1: convert by hand
fixed = {**record, "tags": sorted(record["tags"]), "day": record["day"].isoformat()}
print(json.dumps(fixed))

# option 2: a default function for unknown types
def to_json(obj):
    if isinstance(obj, set):
        return sorted(obj)
    return str(obj)

print(json.dumps(record, default=to_json))
```
</details>

### Cleanup
Run this when you're done to delete the practice folder.

In [ ]:
shutil.rmtree("ex_json", ignore_errors=True)
print(Path("ex_json").exists())